# From Keyword Matching to Real Vector Search

Continuing the Insurellm expert-knowledge-worker project. The previous notebook
used brute-force keyword matching for retrieval -- good for understanding the
basic idea of RAG, but limited: it only works when the user happens to type a
word that exactly matches a dictionary key.

This notebook replaces that with a proper retrieval pipeline:

- **Part A: Chunking** -- splitting documents into overlapping pieces small
  enough to embed and retrieve individually.
- **Part B: Embeddings + Chroma** -- turning each chunk into a vector and
  storing it in a vector database, so retrieval can be based on *meaning*
  rather than exact words.
- **Part C: Visualization** -- reducing those high-dimensional vectors down to
  2D and 3D so I can actually see how the knowledge base clusters.

Cost is still a factor for this fictional company, so sticking with cheap
choices throughout: a low-cost chat model, and a free local embedding model
instead of a paid API.


## Part A: Dividing documents into chunks


In [ ]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go


In [ ]:
# Price is a factor for this company, so sticking with a low-cost model

MODEL = "gpt-4.1-nano"
db_name = "vector_db"
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")


### Getting a feel for the knowledge base size first

Before splitting anything, worth knowing how big the knowledge base actually
is -- both in raw characters and in tokens, since token count is what
determines cost and what fits in a context window.


In [ ]:
# How many characters in all the documents?

knowledge_base_path = "knowledge-base/**/*.md"
files = glob.glob(knowledge_base_path, recursive=True)
print(f"Found {len(files)} files in the knowledge base")

entire_knowledge_base = ""

for file_path in files:
    with open(file_path, 'r', encoding='utf-8') as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

print(f"Total characters in knowledge base: {len(entire_knowledge_base):,}")


In [ ]:
# How many tokens in all the documents?

encoding = tiktoken.encoding_for_model(MODEL)
tokens = encoding.encode(entire_knowledge_base)
token_count = len(tokens)
print(f"Total tokens for {MODEL}: {token_count:,}")


### Loading documents properly with LangChain

Rather than reading files manually like in the previous notebook, using
LangChain's `DirectoryLoader` this time -- it handles loading, and lets me
attach metadata (`doc_type`, based on the subfolder name: employees, products,
contracts, company) to each document as it's loaded. That metadata becomes
useful later for filtering and for coloring the visualization.


In [ ]:
folders = glob.glob("knowledge-base/*")

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")


In [ ]:
documents[1]


### Splitting into chunks

Whole documents are too big to embed and retrieve efficiently, and mixing
unrelated content in one chunk hurts retrieval precision. `RecursiveCharacterTextSplitter`
splits on natural boundaries (paragraphs, then sentences, then words) as a
first choice, falling back to raw character splits only if needed.

`chunk_size=1000` characters per chunk, `chunk_overlap=200` -- the overlap
means adjacent chunks share some content, so a fact split awkwardly across a
chunk boundary is less likely to be lost entirely.


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")


In [ ]:
chunks[100]


## Part B: Turning chunks into vectors, stored in Chroma

**Embeddings** turn text into a list of numbers (a vector) that captures
meaning -- texts with similar meaning end up close together in this vector
space, even if they don't share any exact words. This is the key upgrade over
the previous notebook's keyword matching: retrieval can now work on semantic
similarity instead of literal string matches.

Using `all-MiniLM-L6-v2`, a free local Hugging Face embedding model, to keep
costs down (the OpenAI embedding alternative is commented out below in case I
want to compare quality later). No Hugging Face token should actually be
needed for this particular model, even though I've got `HF_TOKEN` set up from
earlier notebooks just in case.

**Chroma** is the vector database -- it stores each chunk's embedding
alongside the original text and metadata, and supports fast similarity search
over them.


In [ ]:
# Pick an embedding model

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
# embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")


### Peeking at the vectors themselves

Confirming how many vectors got stored, and how many dimensions each one has
-- `all-MiniLM-L6-v2` produces 384-dimensional embeddings, which is on the
smaller/faster end for this kind of model.


In [ ]:
collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")


## Part C: Visualizing the vector store

384 dimensions is impossible to look at directly, so I need to compress it
down to something visual. First, pulling everything out of Chroma along with
its metadata, and assigning a color per document type so the plot is
readable at a glance.


In [ ]:
# Prework

result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['doc_type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]


### Reducing to 2D with t-SNE

t-SNE (t-distributed stochastic neighbor embedding) reduces high-dimensional
vectors down to 2 dimensions while trying to preserve which points were close
together originally -- so this plot is a rough, human-readable map of which
chunks are semantically similar to each other. Hovering over each point shows
its document type and a text preview.


In [ ]:
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()


### The same idea in 3D

Same t-SNE approach, just keeping one more dimension -- worth comparing
against the 2D version to see if clusters that looked overlapping in 2D
actually separate out once given a third dimension to spread into.


In [ ]:
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()


## What I have now, and what's still missing

At this point the knowledge base is chunked, embedded, and browsable visually
-- a real vector store instead of a plain dictionary. What's still missing to
turn this into an actual working assistant again:

- **Retrieval at query time.** So far I've only built and visualized the
  store -- I haven't yet wired up "take a user's question, embed it, find the
  nearest chunks, and feed them into the LLM" the way the brute-force version
  did with its dictionary lookup.
- **A `chat()` function and Gradio UI**, same shape as before, but now backed
  by real similarity search instead of exact keyword matches.

Good next step: bring back the `chat(message, history)` pattern from the
previous notebook, but swap `get_relevant_context` for a call to
`vectorstore.similarity_search(...)`.
